# Format转化

In [3]:
import json
import csv

def parse_reaction_equation(metabolites):
    substrates, products = [], []
    for met_id, coeff in metabolites.items():
        coeff = float(coeff)
        abs_coeff = abs(coeff)
        coeff_str = str(int(abs_coeff)) if abs_coeff == int(abs_coeff) else str(abs_coeff)
        token = met_id if abs_coeff == 1.0 else f"{coeff_str} {met_id}"
        (substrates if coeff < 0 else products).append(token)
    return f"{' + '.join(substrates)} --> {' + '.join(products)}"

def json_to_csv(json_path, csv_path):
    with open(json_path, "r", encoding="utf-8") as f:
        model = json.load(f)
    reactions = model if isinstance(model, list) else model.get("reactions", [])
    reactions = [
    r for r in reactions
    if r.get("annotation", {}).get("sbo") in ("SBO:0000176", "SBO:0000185")
]

    rows = [{"": r["id"], "reaction": r["id"], "equ": parse_reaction_equation(r.get("metabolites", {})), "g0": ""} for r in reactions]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["", "reaction", "equ", "g0"], delimiter='\t')
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ 完成！共 {len(rows)} 条反应 → {csv_path}")
json_to_csv("file/iCZ870.json", "file/iCZ870.csv")

✅ 完成！共 2834 条反应 → file/iCZ870.csv


# G0计算

In [10]:
import math
import re
import pandas as pd
from equilibrator_api import ComponentContribution, Q_
import warnings
warnings.filterwarnings("ignore")

cc = ComponentContribution()
cc.p_h = Q_(7.5)
cc.ionic_strength = Q_("250 mM")
cc.temperature = Q_("310.15 K")

R = 8.31446261815324
T = 310.15

df = pd.read_csv("file/iCZ870.csv", index_col=0,sep="\t")
df["Is Balanced"] = None

def convert_csv_formula(formula):
    formula = formula.replace("-->", "=")

    def replace_metabolite(match):
        coeff = match.group(1)
        met_id = match.group(2)
        base_id = re.sub(r'_[a-z]\d*$', '', met_id)
        if coeff:
            return f"{coeff.strip()} seed:{base_id}"
        else:
            return f"seed:{base_id}"

    formula = re.sub(r'(\d+(?:\.\d+)?\s+)?([a-zA-Z][a-zA-Z0-9]*(?:__[a-zA-Z0-9]+)?_[a-z]\d*)',
                     replace_metabolite, formula)
    return formula

for index, row in df.iterrows():
    equ = row["equ"]
    try:
        converted = convert_csv_formula(equ)
        reaction = cc.parse_reaction_formula(converted)

        is_balanced = reaction.is_balanced()
        df.at[index, "Is Balanced"] = is_balanced
        if is_balanced:
            dG0_prime = cc.standard_dg_prime(reaction)
            dG0_val = dG0_prime.magnitude.nominal_value
            df.at[index, "g0"] = dG0_val
        else:
            df.at[index, "g0"] = 0.0
            print(f"未平衡，跳过: {index} | {converted}")

    except Exception as e:
        print(f"出错: {index} | {converted}")
        print(f"  错误: {str(e)}")

df.to_csv("file/iCZ870_g0.csv",sep="\t")
print(f"完成！共处理 {len(df)} 条，成功 {df['Is Balanced'].sum()} 条")

未平衡，跳过: rxn00058_c_num1 | seed:cpd00007 + 4 seed:cpd00067 + 4 seed:cpd00110 = 2 seed:cpd00001 + 4 seed:cpd00109
未平衡，跳过: rxn00206_c | 2 seed:cpd00067 + 2 seed:cpd00532 = seed:cpd00007 + seed:cpd00025
未平衡，跳过: rxn00224_c | seed:cpd00028 + 2 seed:cpd00067 = seed:cpd01476 + seed:cpd10515
未平衡，跳过: rxn00238_c | seed:cpd00002 + seed:cpd00354 = seed:cpd00008 + seed:cpd00173
未平衡，跳过: rxn00289_c_num1 | seed:cpd02799 = seed:cpd00020 + seed:cpd00036
未平衡，跳过: rxn03061_c | seed:cpd02799 = seed:cpd00001 + seed:cpd02597
未平衡，跳过: rxn00390_c | seed:cpd00001 + seed:cpd01759 = seed:cpd00041 + seed:cpd00049
未平衡，跳过: rxn00622_c_num1 | seed:cpd00001 + seed:cpd02000 = seed:cpd00057 + seed:cpd00067 + seed:cpd00080
未平衡，跳过: rxn00646_c_num1 | seed:cpd00002 + seed:cpd00023 + seed:cpd00084 = seed:cpd00008 + seed:cpd00009 + seed:cpd00067 + seed:cpd00506
未平衡，跳过: rxn00689_c | seed:cpd00002 + seed:cpd00023 + seed:cpd00087 = seed:cpd00008 + seed:cpd00009 + seed:cpd00067 + seed:cpd06227
未平衡，跳过: rxn00758_c_num1 | seed:cpd00001 

In [1]:
import pandas as pd
import json
import re

# 读取文件，指定制表符分隔
icz870_df = pd.read_csv("./file/iCZ870_g0.csv", sep='\t')
icw773_df = pd.read_csv("./file/iCW773_uniprot_modification_del_reaction_g0.csv", sep='\t')
with open("./file/iCZ870.json", "r") as f:
    model = json.load(f)

def strip_numN(name):
    return re.sub(r'_num\d+$', '', name)

def is_reverse(rxn_id):
    return '_reverse' in rxn_id

def base_rxn(rxn_id):
    return rxn_id.replace('_reverse', '')

# 构建JSON映射
rxn_to_bigg = {}
reactions_list = model.get("reactions", [])
for rxn in reactions_list:
    rid = rxn.get("id", "")
    annotation = rxn.get("annotation", {})
    bigg = annotation.get("bigg.reaction", None)
    if bigg is None:
        continue
    if isinstance(bigg, list):
        bigg = bigg[0]
    base = base_rxn(rid)
    if is_reverse(rid):
        rxn_to_bigg[rid] = bigg + "_reverse"
    else:
        rxn_to_bigg[rid] = bigg

# iCW773文件：第1列是reaction，第2列是g0值（列名可能是'equ g0'或其他）
rxn_col_773 = icw773_df.columns[0]  # 第一列是reaction
g0_col_773 = icw773_df.columns[3]   # 第二列是g0值（不管列名叫什么）

icw773_df["_clean_rxn"] = icw773_df[rxn_col_773].apply(strip_numN)
bigg_to_g0 = dict(zip(icw773_df["_clean_rxn"], icw773_df[g0_col_773]))

# iCZ870文件：第1列是Unnamed索引，第2列是reaction，第4列是g0值（列名叫'go'）
rxn_col_870 = icz870_df.columns[1]   # 第二列是reaction
g0_col_870 = "g0"                     # 第四列的列名是'go'，不是'equ g0'

def resolve_g0(raw_rxn_id):
    cleaned = strip_numN(raw_rxn_id)
    reverse = is_reverse(cleaned)
    base = base_rxn(cleaned)
    bigg_base = rxn_to_bigg.get(base, None)
    if bigg_base is None:
        return None
    if bigg_base.endswith("_reverse"):
        bigg_base = bigg_base[:-8]
    target_bigg = (bigg_base + "_reverse") if reverse else bigg_base
    return bigg_to_g0.get(target_bigg, None)

matched = 0
unmatched = []

# 直接替换g0值
for idx, row in icz870_df.iterrows():
    raw_rxn = str(row[rxn_col_870])
    new_g0 = resolve_g0(raw_rxn)

    if new_g0 is not None:
        icz870_df.at[idx, g0_col_870] = new_g0  # 直接替换
        matched += 1
    else:
        unmatched.append(raw_rxn)

# 保存结果
out_path = "./file/iCZ870_g0_n.csv"
icz870_df.to_csv(out_path, sep='\t', index=False)  # 用制表符保存

print(f"完成！总行数:{len(icz870_df)}，成功替换:{matched}，未匹配:{len(unmatched)}")
if unmatched:
    print(f"未匹配示例(前10): {unmatched[:10]}")

完成！总行数:2834，成功替换:508，未匹配:2326
未匹配示例(前10): ['rxn00002_c', 'rxn00011_c', 'rxn00015_c', 'rxn00055_c', 'rxn00058_c_num1', 'rxn00096_c', 'rxn00101_c', 'rxn00105_c_num1', 'rxn00107_c', 'rxn00113_c']


In [6]:
import pandas as pd

df = pd.read_csv("file/iCZ870_g0_n.csv", index_col=0, sep="\t")

target_rxns = [
    "rxn02362_c","rxn02362_c_reverse", #(1) ATP_c + (1) DPG_c + (1) H+_c <=> (1) ADP_c + (1) Phosphate_c + (1) cDPG_c
    'rxn00102_c_reverse', 'rxn00102_c', 'rxn00097_c', 'rxn00097_c_reverse'
]

for rxn_id in target_rxns:
    if rxn_id in df.index:
        df.at[rxn_id, "g0"] = 0.0
        print(f"已将 {rxn_id} 的 g0 设为 0")
    else:
        print(f"警告: {rxn_id} 不在 DataFrame 中，请检查 ID 是否正确")

df.to_csv("file/iCZ870_g0_n.csv", sep="\t")
print("完成！")

已将 rxn02362_c 的 g0 设为 0
已将 rxn02362_c_reverse 的 g0 设为 0
已将 rxn00102_c_reverse 的 g0 设为 0
已将 rxn00102_c 的 g0 设为 0
已将 rxn00097_c 的 g0 设为 0
已将 rxn00097_c_reverse 的 g0 设为 0
完成！


# 处理Kcat

In [2]:
import pandas as pd
import os

# ========== 配置区 ==========
input_file  = 'file/reaction_kcat_MW_modify.csv'   # ← 改成你的文件路径
output_file = 'file/reaction_kcat_MW_modify_n.csv'  # ← 输出文件路径
# ============================

df = pd.read_csv(input_file)

print("转换前预览：")
print(df[['reactions', 'kcat', 'MW']].head())
print(f"\n原始 kcat 范围: {df['kcat'].min():.4f} ~ {df['kcat'].max():.4f}")
print(f"原始 MW   范围: {df['MW'].min():.4f} ~ {df['MW'].max():.4f}")

# 单位转换
df['kcat'] = df['kcat'] * 3600   # 1/s  → 1/h
df['MW']   = df['MW']   / 1000   # Da   → kDa  （或 g/mol → kg/mol，按需调整注释）

# 重新计算 kcat_MW
if 'kcat_MW' in df.columns:
    df['kcat_MW'] = df['kcat'] / df['MW']
    print("\nkcat_MW 已同步更新")

print("\n转换后预览：")
print(df[['reactions', 'kcat', 'MW']].head())
print(f"\n转换后 kcat 范围: {df['kcat'].min():.4f} ~ {df['kcat'].max():.4f}")
print(f"转换后 MW   范围: {df['MW'].min():.4f} ~ {df['MW'].max():.4f}")

df.to_csv(output_file, index=False)
print(f"\n✅ 已保存到: {output_file}")

转换前预览：
                 reactions       kcat            MW
0          rxn05740_c_num1  4187.4453  1.545603e+04
1  rxn05740_c_reverse_num1  3426.0683  1.545603e+04
2               rxn00427_c  2724.2547  5.869408e+04
3            Bio_protein_c  2238.0669  3.504118e+06
4               rxn00799_c  1936.7322  1.990511e+05

原始 kcat 范围: 0.0108 ~ 33000000.0000
原始 MW   范围: 7585.5446 ~ 3504117.9847

kcat_MW 已同步更新

转换后预览：
                 reactions         kcat           MW
0          rxn05740_c_num1  15074803.08    15.456031
1  rxn05740_c_reverse_num1  12333845.88    15.456031
2               rxn00427_c   9807316.92    58.694084
3            Bio_protein_c   8057040.84  3504.117985
4               rxn00799_c   6972235.92   199.051125

转换后 kcat 范围: 38.8800 ~ 118800000000.0000
转换后 MW   范围: 7.5855 ~ 3504.1180

✅ 已保存到: file/reaction_kcat_MW_modify_n.csv


In [7]:
import json
import csv

def extract_metabolites(json_path, csv_path):
    with open(json_path, "r", encoding="utf-8") as f:
        model = json.load(f)

    metabolites = model.get("metabolites", [])

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerow(["id", "lnClb", "lnCub"])
        for met in metabolites:
            met_id = met["id"]
            if met_id.endswith("_e"):
                writer.writerow([met_id, 0, 0])
            else:
                writer.writerow([met_id, -14.50865774, -3.912023005])

    print(f"✅ 完成！共 {len(metabolites)} 个代谢物 → {csv_path}")

extract_metabolites("file/iCZ870.json", "file/iCZ870_met.txt")

✅ 完成！共 1660 个代谢物 → file/iCZ870_met_1.csv
